In [ ]:
import pandas as pd
import json

X = pd.read_parquet("/context/data/X.parquet")
y = pd.read_parquet("/context/data/y.parquet")

with open("/context/data/moons_split.json") as f:
    splits = json.load(f)

X_train = X[X["moon"].isin(splits["train"])]
y_train = y[y["moon"].isin(splits["train"])]

X_test_cloud = X[X["moon"].isin(splits["reduced_cloud"])]
y_test_cloud = y[y["moon"].isin(splits["reduced_cloud"])]


In [ ]:
import joblib
import os
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, rankdata
from sklearn.linear_model import Ridge
from sklearn.preprocessing import QuantileTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from lightgbm import LGBMRegressor


def train(X_train, y_train, model_directory_path, loop_moon, embargo):
    # ── Config ────────────────────────────────────────────────────────────────
    N_FEATURES = 200
    MAX_IC_MOONS = 80   # sample this many moons for feature selection speed

    # ── Utilities ──────────────────────────────────────────────────────────────
    def cs_rank(series):
        n = len(series)
        if n < 2:
            return series * 0.0
        r = series.rank(method="average")
        return (r - 1) / (n - 1) - 0.5

    def fill_median(df, cols):
        df = df.copy()
        for c in cols:
            if c in df.columns:
                med = df[c].median()
                df[c] = df[c].fillna(med if not np.isnan(med) else 0.0)
        return df

    def rank_normalise(df, col="prediction"):
        df = df.copy()
        n = len(df)
        if n < 2:
            return df
        df[col] = (df[col].rank(method="average") - 1) / (n - 1) - 0.5
        return df

    # ── Feature selection: LOO cross-sectional IC ──────────────────────────────
    def select_features(X, y_series, top_n, max_moons):
        all_feats = [c for c in X.columns if c.startswith("Feature_")]
        moons = X["moon"].values
        unique_moons = np.unique(moons)
        if len(unique_moons) > max_moons:
            rng = np.random.default_rng(42)
            unique_moons = rng.choice(unique_moons, size=max_moons, replace=False)

        ic_sums  = np.zeros(len(all_feats))
        ic_counts = np.zeros(len(all_feats))
        for m in unique_moons:
            mask = moons == m
            yt = y_series.values[mask]
            if yt.std() < 1e-10:
                continue
            for j, f in enumerate(all_feats):
                xf = X[f].values[mask]
                if xf.std() < 1e-10:
                    continue
                r, _ = pearsonr(xf, yt)
                if not np.isnan(r):
                    ic_sums[j]   += r
                    ic_counts[j] += 1

        counts_safe = np.where(ic_counts > 0, ic_counts, 1)
        mean_ics = ic_sums / counts_safe
        mean_ics[ic_counts == 0] = 0.0

        top_idx = np.argsort(np.abs(mean_ics))[::-1][:top_n]
        selected = [all_feats[i] for i in top_idx]
        print(f"  Feature selection done. Top 5: {selected[:5]}")
        return selected

    # ── Prepare training data ──────────────────────────────────────────────────
    df = X_train.merge(y_train[["moon", "id", "target"]], on=["moon", "id"])
    df = df.dropna(subset=["target"])

    print(f"  Selecting top {N_FEATURES} features from {X_train['moon'].nunique()} moons ...")
    feature_cols = select_features(
        df.drop(columns=["target"]),
        df["target"],
        top_n=N_FEATURES,
        max_moons=MAX_IC_MOONS,
    )

    df = fill_median(df, feature_cols)
    df["target_rank"] = df.groupby("moon")["target"].transform(cs_rank)

    Xf = df[feature_cols]
    yf = df["target_rank"]

    # ── Fit 3 models with stronger regularization ──────────────────────────────
    print("  Fitting Ridge (alpha=100, increased from 30) ...")
    qt = QuantileTransformer(output_distribution="normal", random_state=42)
    Xf_qt = qt.fit_transform(Xf)
    ridge = Ridge(alpha=100.0)  # Increased regularization to prevent overfitting
    ridge.fit(Xf_qt, yf)

    print("  Fitting LightGBM (increased regularization) ...")
    lgbm = LGBMRegressor(
        n_estimators=400, learning_rate=0.015, num_leaves=31,  # Reduced from 63
        min_child_samples=100, subsample=0.5, colsample_bytree=0.5,  # Increased min_samples, reduced subsample
        reg_alpha=1.0, reg_lambda=5.0, random_state=42, n_jobs=-1, verbose=-1,  # 10x stronger regularization
        lambda_l1=1.0, lambda_l2=5.0  # Additional L1/L2 penalty
    )
    lgbm.fit(Xf, yf)

    print("  Fitting HistGBM (increased regularization) ...")
    hgbm = HistGradientBoostingRegressor(
        max_iter=300, max_leaf_nodes=31, max_bins=128,  # Reduced max_leaf_nodes from 63
        l2_regularization=1.0, learning_rate=0.02,  # 10x stronger L2 regularization
        early_stopping=False, random_state=2022,
    )
    hgbm.fit(Xf_qt, yf)

    # ── IC-weighted blend from validation window ───────────────────────────────
    moons     = sorted(df["moon"].unique())
    n_val     = max(4, len(moons) // 5)  # Use 20% of moons for validation (more stable)
    val_moons = moons[-n_val:]
    dv        = df[df["moon"].isin(val_moons)]
    Xv        = dv[feature_cols]
    Xv_qt     = qt.transform(Xv)
    yv        = dv["target_rank"].values

    def ic(yt, yp):
        if len(yt) < 5: return 0.0
        r, _ = pearsonr(yp, yt)
        return float(r) if not np.isnan(r) else 0.0

    w = np.array([
        max(ic(yv, ridge.predict(Xv_qt)), 0),
        max(ic(yv, lgbm.predict(Xv)),    0),
        max(ic(yv, hgbm.predict(Xv_qt)), 0),
    ])
    w = w / w.sum() if w.sum() > 0 else np.array([0.33, 0.34, 0.33])
    print(f"  Blend weights → Ridge:{w[0]:.3f}  LGBM:{w[1]:.3f}  HGBM:{w[2]:.3f}")

    joblib.dump(
        {"ridge": ridge, "lgbm": lgbm, "hgbm": hgbm,
         "qt": qt, "features": feature_cols, "weights": w},
        os.path.join(model_directory_path, "model.joblib"),
    )
    print("  Model saved.")

In [ ]:
def infer(X_test, model_directory_path, loop_moon, embargo):

    obj      = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    features = obj["features"]
    w        = obj["weights"]

    X = X_test.copy()
    for c in features:
        if c in X.columns:
            med = X[c].median()
            X[c] = X[c].fillna(med if not np.isnan(med) else 0.0)

    Xf    = X[features]
    Xf_qt = obj["qt"].transform(Xf)

    # Get predictions from each model
    ridge_pred = obj["ridge"].predict(Xf_qt)
    lgbm_pred = obj["lgbm"].predict(Xf)
    hgbm_pred = obj["hgbm"].predict(Xf_qt)
    
    # Rank predictions individually to normalize scale
    ridge_ranked = rankdata(ridge_pred)
    lgbm_ranked = rankdata(lgbm_pred)
    hgbm_ranked = rankdata(hgbm_pred)
    
    # Apply weighted blend on ranked predictions
    blended = (
        w[0] * ridge_ranked +
        w[1] * lgbm_ranked +
        w[2] * hgbm_ranked
    )

    # Final rank normalization
    n = len(blended)
    final = (rankdata(blended) - 1) / max(n - 1, 1) - 0.5
    
    # Clip to avoid extreme predictions (additional regularization)
    final = np.clip(final, -0.5, 0.5)

    return pd.DataFrame({
        "moon":       X_test["moon"].values,
        "id":         X_test["id"].values,
        "prediction": final,
    })

In [ ]:
embargo = 4
model_dir = "./model_cr2"
os.makedirs(model_dir, exist_ok=True)

train(X_train, y_train, model_dir, loop_moon=0, embargo=embargo)

results = []
for moon in splits["reduced_cloud"]:
    X_moon = X_test_cloud[X_test_cloud["moon"] == moon]
    pred = infer(X_moon, model_dir, loop_moon=moon, embargo=embargo)
    results.append(pred)

predictions = pd.concat(results)

from scipy.stats import pearsonr

for moon in splits["reduced_cloud"]:
    p  = predictions[predictions["moon"] == moon]
    gt = y_test_cloud[y_test_cloud["moon"] == moon]
    merged = p.merge(gt, on=["moon", "id"])
    if len(merged) > 1:
        corr, _ = pearsonr(merged["prediction"], merged["target"])
        print(f"Moon {moon}: Pearson r = {corr:.4f}")
